# GOAT $U_{XX}$ gate

**Ansatz**  
one slow envelope per carrier line.  
carriers are chosen from the Ising-frame 2P1e lines
$$\omega_e(s_1,s_2)=|\gamma_e|B_0-\frac{s_1A_1+s_2A_2}{2},\quad
\omega_{n1}(s_e)=\gamma_nB_0-\frac{s_eA_1}{2},\quad
\omega_{n2}(s_e)=\gamma_nB_0-\frac{s_eA_2}{2}.$$
optimization is done in a carrier-resolved effective model, i.e. one channel per line, not raw $\cos(\omega t)$ in the lab frame.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt
from collections import OrderedDict
from qutip_qoc.objective import Objective
from qutip_qoc.pulse_optim import optimize_pulses

np.set_printoptions(precision=6,suppress=True)

## ops

In [2]:
I2=qt.qeye(2)
sx=0.5*qt.sigmax()
sy=0.5*qt.sigmay()
sz=0.5*qt.sigmaz()

def op_on(q,op):
    ops=[I2,I2,I2]
    ops[q]=op
    return qt.tensor(ops)

Sx,Sy,Sz=op_on(0,sx),op_on(0,sy),op_on(0,sz)
Ix1,Iy1,Iz1=op_on(1,sx),op_on(1,sy),op_on(1,sz)
Ix2,Iy2,Iz2=op_on(2,sx),op_on(2,sy),op_on(2,sz)

def ket(e,n1,n2):
    return qt.tensor(qt.basis(2,e),qt.basis(2,n1),qt.basis(2,n2))

def x_from(T):
    return T+T.dag()

def y_from(T):
    return -1j*(T-T.dag())

## target

In [3]:
UXX=0.5*np.array([
    [1,0,0,1,1,0,0,-1],
    [0,1,1,0,0,1,-1,0],
    [0,1,1,0,0,-1,1,0],
    [1,0,0,1,-1,0,0,1],
    [1,0,0,-1,1,0,0,1],
    [0,1,-1,0,0,1,1,0],
    [0,-1,1,0,0,1,1,0],
    [-1,0,0,1,1,0,0,1],
],dtype=complex)

U0=qt.qeye([2,2,2])
U_target=qt.Qobj(UXX,dims=[[2,2,2],[2,2,2]])
print("unitarity err =",np.linalg.norm((U_target.dag()*U_target-U0).full()))

unitarity err = 0.0


## lines

In [4]:
A1=95
A2=9
gamma_e=27.97e3
gamma_n=17.23
B0=1.33

w_e={}
for s1 in [1,-1]:
    for s2 in [1,-1]:
        w_e[(s1,s2)]=abs(gamma_e)*B0-(s1*A1+s2*A2)/2

w_n1={1:gamma_n*B0-A1/2,-1:gamma_n*B0+A1/2}
w_n2={1:gamma_n*B0-A2/2,-1:gamma_n*B0+A2/2}
w_mix=A1+A2+gamma_n*B0

print("ESR [MHz]")
for k,v in w_e.items():
    print(k,v)
print("NMR1 [MHz]",w_n1)
print("NMR2 [MHz]",w_n2)
print("mix [MHz]",w_mix)

ESR [MHz]
(1, 1) 37148.1
(1, -1) 37157.1
(-1, 1) 37243.1
(-1, -1) 37252.1
NMR1 [MHz] {1: -24.5841, -1: 70.4159}
NMR2 [MHz] {1: 18.4159, -1: 27.4159}
mix [MHz] 126.9159


## effective model

In [5]:
H_drift=2*np.pi*(A1*Sz*Iz1+A2*Sz*Iz2)

def e_line_ops(n1,n2):
    T=ket(0,n1,n2)*ket(1,n1,n2).dag()
    return x_from(T),y_from(T)

def n1_line_ops(e):
    T=(ket(e,0,0)*ket(e,1,0).dag()
      +ket(e,0,1)*ket(e,1,1).dag())
    return x_from(T),y_from(T)

def n2_line_ops(e):
    T=(ket(e,0,0)*ket(e,0,1).dag()
      +ket(e,1,0)*ket(e,1,1).dag())
    return x_from(T),y_from(T)

control_ops=[]
amp_bounds={}

for n1 in [0,1]:
    for n2 in [0,1]:
        s1=1 if n1==0 else -1
        s2=1 if n2==0 else -1
        Qx,Qy=e_line_ops(n1,n2)
        control_ops.append((f"e_{s1:+d}{s2:+d}_x",Qx,w_e[(s1,s2)]))
        control_ops.append((f"e_{s1:+d}{s2:+d}_y",Qy,w_e[(s1,s2)]))
        amp_bounds[f"e_{s1:+d}{s2:+d}_x"]=8
        amp_bounds[f"e_{s1:+d}{s2:+d}_y"]=8

for e in [0,1]:
    se=1 if e==0 else -1
    Qx,Qy=n1_line_ops(e)
    control_ops.append((f"n1_{se:+d}_x",Qx,w_n1[se]))
    control_ops.append((f"n1_{se:+d}_y",Qy,w_n1[se]))
    amp_bounds[f"n1_{se:+d}_x"]=3
    amp_bounds[f"n1_{se:+d}_y"]=3

for e in [0,1]:
    se=1 if e==0 else -1
    Qx,Qy=n2_line_ops(e)
    control_ops.append((f"n2_{se:+d}_x",Qx,w_n2[se]))
    control_ops.append((f"n2_{se:+d}_y",Qy,w_n2[se]))
    amp_bounds[f"n2_{se:+d}_x"]=3
    amp_bounds[f"n2_{se:+d}_y"]=3

print("n_ctrl =",len(control_ops))

n_ctrl = 16


## GOAT ansatz

In [6]:
T=4
K=2
n_ts=160
tlist=np.linspace(0,T,n_ts)
n_params_per_ctrl=2*K+1

def edge(t):
    return np.sin(np.pi*t/T)**2

def basis_val(t,idx):
    if idx==0:
        return 1
    k=(idx+1)//2
    if idx%2==1:
        return np.cos(2*np.pi*k*t/T)
    return np.sin(2*np.pi*k*t/T)

def env(t,p):
    out=0
    for idx in range(len(p)):
        out+=p[idx]*basis_val(t,idx)
    return edge(t)*out

def env_grad(t,p,idx):
    return edge(t)*basis_val(t,idx)

H=[H_drift]
for _,Q,_ in control_ops:
    H.append([Q,env,{"grad":env_grad}])

obj=Objective(initial=U0,H=H,target=U_target)

## run

In [7]:
def make_control_params(seed,base_scale=0.08):
    rng=np.random.default_rng(seed)
    params=OrderedDict()
    for name,_,_ in control_ops:
        Amax=amp_bounds[name]
        guess=base_scale*Amax*rng.standard_normal(n_params_per_ctrl)
        guess[0]=0
        params[name]={
            "guess":guess,
            "bounds":[(-Amax,Amax)]*n_params_per_ctrl,
        }
    return params

def run_goat(seed,max_iter=120,maxfun=250,base_scale=0.08):
    params=make_control_params(seed,base_scale=base_scale)
    result=optimize_pulses(
        objectives=[obj],
        control_parameters=params,
        tlist=tlist,
        algorithm_kwargs={
            "alg":"GOAT",
            "fid_type":"PSU",
            "fid_err_targ":1e-4,
            "max_iter":max_iter,
        },
        minimizer_kwargs={
            "method":"L-BFGS-B",
            "options":{
                "maxiter":max_iter,
                "maxfun":maxfun,
                "ftol":1e-9,
                "gtol":1e-6,
            },
        },
        integrator_kwargs={
            "method":"dop853",
            "atol":1e-7,
            "rtol":1e-6,
            "nsteps":20000,
        },
    )
    return result

seeds=[0,1]
all_results=[]
for seed in seeds:
    print("seed",seed)
    res=run_goat(seed)
    all_results.append((seed,res))
    print("infid",float(res.infidelity))

best_seed,best_result=min(all_results,key=lambda x:float(x[1].infidelity))
print("best seed",best_seed)
print("best infid",float(best_result.infidelity))

seed 0


## plot

In [ ]:
plt.figure(figsize=(10,5))
for i,(name,_,w) in enumerate(control_ops):
    y=np.array(best_result.optimized_controls[i])
    if np.max(np.abs(y))>1e-3:
        plt.plot(tlist,y,label=f"{name}  {w:.3f} MHz")
plt.xlabel("t [us]")
plt.ylabel("env")
plt.grid(alpha=0.3)
plt.legend(ncol=2,fontsize=8)
plt.show()